# Customer Churn Prediction Pipeline

In [0]:
import os
import numpy as np
import matplotlib.pyplot as plt
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql import SparkSession
from pyspark.ml.feature import StringIndexer
from pyspark.ml.stat import Correlation
from pyspark.ml.feature import VectorAssembler

spark = (
    SparkSession.builder
    .appName("ChurnAnalysis")
    .config("spark.sql.pushdown.enabled", "true")
    .getOrCreate()
)

BASE_PATH = "/Volumes/workspace/default/customer_data"
RAW_DIR = f"{BASE_PATH}/raw"
PARTITIONED_DIR = f"{BASE_PATH}/partitioned"
CHURN_CSV = f"{RAW_DIR}/churn.csv"

In [0]:
churn_csv = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("mode", "PERMISSIVE")
    .csv(CHURN_CSV)
)

print("Rows:", churn_csv.count())
churn_csv.printSchema()
display(churn_csv.limit(10))


In [0]:
TARGET_SAMPLE_SIZE = 10000

total_count = churn_csv.count()
counts_per_class = churn_csv.groupBy("Exited").count().collect()

sample_ratio = min(1.0, TARGET_SAMPLE_SIZE / total_count)
fractions = {row["Exited"]: sample_ratio for row in counts_per_class}

# Stratified sample using native expressions (rand + comparison) instead of
# sampleBy, which compiles to a Scala UDF filter that Photon cannot accelerate.
rand_col = "__sample_rand"
keep_row = F.lit(False)
for stratum, fraction in fractions.items():
    keep_row = keep_row | ((F.col("Exited") == stratum) & (F.col(rand_col) < fraction))

some_churn = (
    churn_csv
    .withColumn(rand_col, F.rand(seed=69))
    .where(keep_row)
    .drop(rand_col)
)

In [0]:
some_churn.groupBy("Exited").agg(
    F.round(F.mean("Age"), 1).alias("avgAge"),
    F.round(F.mean("IsActiveMember") * 100, 1).alias("PctActiveMembers"),
    F.round(F.mean("HasCrCard") * 100, 1).alias("PctCrCardHolders"),
    F.round(F.mean("NumOfProducts"), 2).alias("avgNumOfProducts"),
    F.round(F.mean("Balance"), 2).alias("avgBalance"),
    F.round(F.mean("EstimatedSalary"), 2).alias("avgEstimatedSalary"),
    F.round(F.mean("Tenure"), 1).alias("avgTenure")
).show()

some_churn.groupBy("Geography", "Gender").agg(
    F.count("*").alias("TotalCustomers"),
    F.round(F.mean("Balance"),1).alias("avgBalance"),
    F.round(F.mean("CreditScore")).alias("avgScore").cast('int'),
    F.sum("Exited").alias("ChurnedCount")
).sort(F.desc('avgBalance')).show()


In [0]:

# 1. Exclude ID / identifier columns
excluded_cols = {"RowNumber", "CustomerId", "Surname"}
candidate_cols = [c for c in churn_csv.columns if c not in excluded_cols]

# 2. Encode categorical string columns (Geography, Gender) to numeric for correlation
df_encoded = churn_csv
feature_cols = []

for c, dtype in churn_csv.select(candidate_cols).dtypes:
    if dtype == "string":
        indexed_col = f"{c}_idx"
        indexer = StringIndexer(inputCol=c, outputCol=indexed_col)
        df_encoded = indexer.fit(df_encoded).transform(df_encoded)
        feature_cols.append(indexed_col)
    else:
        feature_cols.append(c)

# 3. Assemble features into a vector and compute Pearson correlation in Spark
assembler = VectorAssembler(inputCols=feature_cols, outputCol="corr_features")
vector_df = assembler.transform(df_encoded).select("corr_features")

corr_matrix = Correlation.corr(vector_df, "corr_features", method="pearson").collect()[0][0].toArray()

# 4. Clean display labels (strip '_idx')
display_labels = [c.replace("_idx", "") for c in feature_cols]
n = len(display_labels)

# 5. Plot Heatmap using pure Matplotlib
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr_matrix, cmap="coolwarm", vmin=-1, vmax=1)

# Set tick labels
ax.set_xticks(np.arange(n))
ax.set_yticks(np.arange(n))
ax.set_xticklabels(display_labels, rotation=45, ha="right")
ax.set_yticklabels(display_labels)

# Add correlation coefficients inside cells
for i in range(n):
    for j in range(n):
        val = corr_matrix[i, j]
        text_color = "white" if abs(val) > 0.5 else "black"
        ax.text(j, i, f"{val:.2f}", ha="center", va="center", color=text_color, fontsize=8)

fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Pearson Correlation")
ax.set_title("Feature Correlation Heatmap", fontsize=13, pad=12)

plt.tight_layout()
plt.show()

In [0]:
age_bucket_df = (
    some_churn
    .withColumn("age_bin_start", (F.floor(F.col("Age") / 5) * 5).cast("int"))
    .groupBy("age_bin_start")
    .agg(
        F.count("*").alias("total"),
        F.sum(F.when(F.col("Exited") == 1, 1).otherwise(0)).alias("churned_count"),
        F.sum(F.when(F.col("Exited") == 0, 1).otherwise(0)).alias("retained_count"),
    )
    .withColumn("churn_pct", (F.col("churned_count") / F.col("total")) * 100)
    .withColumn("retained_pct", (F.col("retained_count") / F.col("total")) * 100)
    .orderBy("age_bin_start")
)

# 2. Collect only the aggregated summary rows to the driver (< 20 rows)
agg_pdf = age_bucket_df.toPandas()

# 3. 100% Stacked Bar Chart with Matplotlib
fig, ax = plt.subplots(figsize=(10, 4.5))

x_labels = [f"{int(start)}-{int(start + 4)}" for start in agg_pdf["age_bin_start"]]
x_pos = range(len(x_labels))

ax.bar(
    x_pos, 
    agg_pdf["churn_pct"], 
    label="Churned (1)", 
    color='red',
    alpha=0.5,
    width=0.65
)
ax.bar(
    x_pos, 
    agg_pdf["retained_pct"], 
    bottom=agg_pdf["churn_pct"], 
    label="Retained (0)", 
    color='green',
    alpha=0.5,
    width=0.65
)

# Formatting
ax.set_xticks(x_pos)
ax.set_xticklabels(x_labels, rotation=45)
ax.set_xlabel("Age Group")
ax.set_ylabel("Percentage (%)")
ax.set_ylim(0, 100)
ax.set_title("Retention vs. Churn Rate by 5-Year Age Group (100% Stacked)")
ax.legend(loc="upper left")
ax.grid(axis="y", linestyle="--", alpha=0.3)

plt.tight_layout()
plt.show()

In [0]:
some_churn.createOrReplaceTempView("churn")

In [0]:
from matplotlib import ticker

sql_query = """
    SELECT 
        Balance,
        IsActiveMember,
        ROUND(
            AVG(Exited) OVER (
                PARTITION BY IsActiveMember
                ORDER BY Balance
                ROWS BETWEEN 500 PRECEDING AND 500 FOLLOWING
            ) * 100.0, 
            2
        ) AS rolling_churn_rate
    FROM churn
    WHERE Balance > 0
    ORDER BY Balance
"""

plot_pdf = spark.sql(sql_query).toPandas()
# 2. Plot
fig, ax = plt.subplots(figsize=(10, 5))

for active_status, label, color in [
    (0, "Inactive Member (0)", "#d95f02"),
    (1, "Active Member (1)", "#2b5c8f")
]:
    sub = plot_pdf[plot_pdf["IsActiveMember"] == active_status]
    ax.plot(
        sub["Balance"] * 1e-3, 
        sub["rolling_churn_rate"], 
        label=label, 
        color=color, 
        linewidth=2
    )

ax.set_ylim(10, 40)
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, pos: f"{x:.0f}k"))
ax.set_xlabel("Balance")
ax.set_ylabel("Churn Probability (%)")
ax.set_title("Continuous Churn Rate by Balance (Split by Member Activity)")
ax.legend(loc="upper right")
ax.grid(True, linestyle="--", alpha=0.3)

plt.tight_layout()
plt.show()

In [0]:
# Compute churn rate directly in Spark
prod_churn = (
    some_churn.groupBy("NumOfProducts")
    .agg(
        F.count("*").alias("Total"),
        (F.mean("Exited") * 100).alias("ChurnRate")
    )
    .orderBy("NumOfProducts")
    .toPandas()
)

fig, ax1 = plt.subplots(figsize=(8, 4.5))

# Bar chart: Total Customers
color = "#2b5c8f"
ax1.set_xlabel("Number of Products")
ax1.set_ylabel("Total Customers", color=color)
bars = ax1.bar(prod_churn["NumOfProducts"], prod_churn["Total"], color=color, alpha=0.6, width=0.5)
ax1.tick_params(axis="y", labelcolor=color)

# Line chart: Churn Rate (%)
ax2 = ax1.twinx()
color = "#d95f02"
ax2.set_ylabel("Churn Rate (%)", color=color)
ax2.plot(prod_churn["NumOfProducts"], prod_churn["ChurnRate"], color=color, marker="o", linewidth=2.5)
ax2.tick_params(axis="y", labelcolor=color)

plt.title("Customer Volume vs. Churn Rate by Number of Products")
plt.tight_layout()
plt.show()

In [0]:
spark.sql("""
    SELECT 
        Geography, 
        COUNT(*) AS TotalCustomers, 
        ROUND(SUM(Exited)/TotalCustomers*100, 1) AS PercentLost
    FROM churn
    GROUP BY Geography
    ORDER BY PercentLost DESC
""").show()

spark.sql("""
    SELECT 
        Gender,
        COUNT(*) AS TotalCustomers,
        ROUND(SUM(Exited)/TotalCustomers*100, 1) AS PercentLost
    FROM churn
    GROUP BY Gender
    ORDER BY PercentLost DESC;
""").show()